In [1]:
# PHASE 0 - Cell 1: Install required libraries for Vaultify
!pip install -q qdrant-client
!pip install -q groq
!pip install -q docling
!pip install -q "rapidocr<3.8"
!pip install -q onnxruntime
!pip install -q mcp[cli]
!pip install -q flask
!pip install -q pyngrok
!pip install -q python-dotenv
!pip install -q sentence-transformers

print("✅ Vaultify base libraries installed successfully.")

✅ Vaultify base libraries installed successfully.


In [24]:
# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!sudo dpkg -i cloudflared-linux-amd64.deb
!rm cloudflared-linux-amd64.deb
print("✅ cloudflared installed")

Selecting previously unselected package cloudflared.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.6.1) ...
Setting up cloudflared (2026.6.1) ...
Processing triggers for man-db (2.10.2-1) ...
✅ cloudflared installed


In [2]:
# PHASE 0 - Cell 2: Read API keys and tokens from Colab Secrets
from google.colab import userdata

QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
NGROK_TOKEN = userdata.get('NGROK_TOKEN')

# Verify all secrets are present (never print actual values for security)
checks = {
    "QDRANT_URL": QDRANT_URL,
    "QDRANT_API_KEY": QDRANT_API_KEY,
    "GROQ_API_KEY": GROQ_API_KEY,
    "NGROK_TOKEN": NGROK_TOKEN,
}

for name, value in checks.items():
    status = "✅ Found" if value else "❌ MISSING"
    print(f"{name}: {status}")

QDRANT_URL: ✅ Found
QDRANT_API_KEY: ✅ Found
GROQ_API_KEY: ✅ Found
NGROK_TOKEN: ✅ Found


In [3]:
# PHASE 0 - Cell 3: Test Qdrant Cloud connection
from qdrant_client import QdrantClient

try:
    qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    collections = qdrant.get_collections()
    print("✅ Qdrant Cloud connection successful.")
    print(f"Existing collections: {collections.collections}")
except Exception as e:
    print(f"❌ Qdrant connection error: {e}")

✅ Qdrant Cloud connection successful.
Existing collections: [CollectionDescription(name='vaultify_documents')]


In [4]:
# PHASE 0 - Cell 4: Test Groq Cloud connection
from groq import Groq

try:
    groq_client = Groq(api_key=GROQ_API_KEY)
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Answer in one word: is this test successful?"}],
        max_tokens=10,
    )
    print("✅ Groq Cloud connection successful.")
    print(f"Model response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Groq connection error: {e}")

✅ Groq Cloud connection successful.
Model response: Yes.


In [5]:
# PHASE 0 - Cell 5: Test Ngrok tunnel connectivity
from pyngrok import ngrok

try:
    ngrok.set_auth_token(NGROK_TOKEN)
    test_tunnel = ngrok.connect(8000)
    print("✅ Ngrok tunnel opened successfully.")
    print(f"Test URL: {test_tunnel.public_url}")

    ngrok.disconnect(test_tunnel.public_url)
    print("Tunnel closed (this was just a test).")
except Exception as e:
    print(f"❌ Ngrok connection error: {e}")

✅ Ngrok tunnel opened successfully.
Test URL: https://rotten-tipping-laborer.ngrok-free.dev
Tunnel closed (this was just a test).


In [6]:
# PHASE 1 - Cell 0: Check if documents are already indexed in Qdrant
# If we have 694+ chunks (615 Apple + 79 Tesla), skip the entire Phase 1 pipeline

from qdrant_client import QdrantClient

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
COLLECTION_NAME = "vaultify_documents"

SKIP_PHASE_1 = False

try:
    # Check if collection exists
    collections = [c.name for c in qdrant.get_collections().collections]

    if COLLECTION_NAME in collections:
        # Count points for each user
        apple_count = qdrant.count(
            collection_name=COLLECTION_NAME,
            count_filter={"must": [{"key": "user_id", "match": {"value": "test_user_apple_demo"}}]}
        ).count

        tesla_count = qdrant.count(
            collection_name=COLLECTION_NAME,
            count_filter={"must": [{"key": "user_id", "match": {"value": "test_user_tesla_demo"}}]}
        ).count

        total = apple_count + tesla_count

        print(f"📊 Existing data in Qdrant:")
        print(f"   Apple chunks: {apple_count}")
        print(f"   Tesla chunks: {tesla_count}")
        print(f"   Total: {total}")

        # If we have at least 694 chunks (615 Apple + 79 Tesla), skip Phase 1
        if total >= 694:
            print(f"\n✅ Data already exists! Skipping Phase 1 pipeline.")
            SKIP_PHASE_1 = True
        else:
            print(f"\n⚠️ Insufficient data ({total}/694). Running Phase 1...")
            SKIP_PHASE_1 = False
    else:
        print(f"⚠️ Collection '{COLLECTION_NAME}' not found. Running Phase 1...")
        SKIP_PHASE_1 = False

except Exception as e:
    print(f"❌ Error checking Qdrant: {e}")
    print("Running Phase 1 to be safe...")
    SKIP_PHASE_1 = False

print(f"\nSKIP_PHASE_1 = {SKIP_PHASE_1}")

📊 Existing data in Qdrant:
   Apple chunks: 1230
   Tesla chunks: 79
   Total: 1309

✅ Data already exists! Skipping Phase 1 pipeline.

SKIP_PHASE_1 = True


In [7]:
# PHASE 1 - Cell 1: Download test documents for multi-tenant testing
# NOTE: This cell is for DEVELOPMENT/TESTING only.
# In production, users upload documents via the Flask dashboard (/upload route).
# We download two PDFs here to simulate two different companies:
#   - company_a: Apple 2025 Form 10-K (financial report, ~100 pages)
#   - company_b: Tesla Q4 2025 Update (quarterly report, ~34 pages)

import urllib.request
import os

# Test document 1: Apple 2025 Form 10-K (for company_a)
apple_url = "https://s2.q4cdn.com/470004039/files/doc_financials/2025/ar/_10-K-2025-As-Filed.pdf"
apple_path = "/content/apple_10k_2025.pdf"

# Test document 2: Tesla Q4 2025 Update (for company_b)
tesla_url = "https://assets-ir.tesla.com/tesla-contents/IR/TSLA-Q4-2025-Update.pdf"
tesla_path = "/content/TSLA-Q4-2025-Update.pdf"

# Download Apple 10-K
print("📥 Downloading Apple 2025 Form 10-K...")
urllib.request.urlretrieve(apple_url, apple_path)
apple_size_mb = os.path.getsize(apple_path) / (1024 * 1024)
print(f"✅ Apple PDF downloaded: {apple_path} ({apple_size_mb:.2f} MB)")

# Download Tesla Q4 2025 Update
print("\n📥 Downloading Tesla Q4 2025 Update...")
urllib.request.urlretrieve(tesla_url, tesla_path)
tesla_size_mb = os.path.getsize(tesla_path) / (1024 * 1024)
print(f"✅ Tesla PDF downloaded: {tesla_path} ({tesla_size_mb:.2f} MB)")

print(f"\n📁 Both test documents are ready in /content/")

📥 Downloading Apple 2025 Form 10-K...
✅ Apple PDF downloaded: /content/apple_10k_2025.pdf (1.77 MB)

📥 Downloading Tesla Q4 2025 Update...
✅ Tesla PDF downloaded: /content/TSLA-Q4-2025-Update.pdf (5.39 MB)

📁 Both test documents are ready in /content/


In [8]:
# PHASE 1 - Cell 2: Parse both PDFs with Docling (GPU-aware)
# We process both documents in a loop and store their markdown content
# in a dictionary. This makes the pipeline scalable for future documents.
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.datamodel.base_models import InputFormat
import time
import torch

# Check if GPU is available
print(f"CUDA available: {torch.cuda.is_available()}")

# Configure Docling to use GPU if available
pipeline_options = PdfPipelineOptions()
if torch.cuda.is_available():
    pipeline_options.accelerator_options = AcceleratorOptions(
        num_threads=4,
        device=AcceleratorDevice.CUDA
    )

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

# Define the documents to parse
documents = {
    "apple_10k_2025.pdf": "/content/apple_10k_2025.pdf",
    "TSLA-Q4-2025-Update.pdf": "/content/TSLA-Q4-2025-Update.pdf"
}

parsed_docs = {}

for filename, filepath in documents.items():
    print(f"\n🚀 Starting conversion for {filename}...")
    start_time = time.time()

    result = converter.convert(filepath)

    elapsed = time.time() - start_time
    print(f"✅ {filename} converted in {elapsed:.1f} seconds.")

    # Export to Markdown and store in dictionary
    parsed_docs[filename] = result.document.export_to_markdown()

print(f"\n📊 Parsing Summary:")
for filename, md in parsed_docs.items():
    print(f" - {filename}: {len(md)} characters")

CUDA available: True

🚀 Starting conversion for apple_10k_2025.pdf...


[INFO] 2026-07-06 11:51:34,905 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-06 11:51:34,921 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-07-06 11:51:34,922 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-07-06 11:51:35,035 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-06 11:51:35,040 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-07-06 11:51:35,042 [RapidOCR] main.py:53: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-07-06 11:51:35,095 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-07-06 11:51:35,126 [RapidOCR] download_file.py:60: File exists and is valid: /usr/loc

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

✅ apple_10k_2025.pdf converted in 51.2 seconds.

🚀 Starting conversion for TSLA-Q4-2025-Update.pdf...


[WARNING] 2026-07-06 11:52:37,033 [RapidOCR] main.py:125: The text detection result is empty


✅ TSLA-Q4-2025-Update.pdf converted in 67.8 seconds.

📊 Parsing Summary:
 - apple_10k_2025.pdf: 342871 characters
 - TSLA-Q4-2025-Update.pdf: 71727 characters


In [9]:
# PHASE 1 - Cell 3: Smart chunking for all parsed documents
# We use a table-aware chunking strategy: tables are kept intact with their
# preceding header for context, while plain text is split into fixed-size
# pieces with overlap to preserve semantic meaning.

def smart_chunk_markdown(markdown_text, chunk_size=500, overlap=50):
    """
    Chunk markdown text while keeping tables intact.
    Tables are kept as whole chunks with their preceding header for context.
    Plain text is split into fixed-size pieces with overlap.
    """
    chunks = []

    # Split the document into lines
    lines = markdown_text.split('\n')

    current_header = ""
    buffer = []

    def flush_text_buffer(buffer_lines):
        """Chunk plain text buffer into fixed-size pieces with overlap."""
        text = '\n'.join(buffer_lines).strip()
        if not text:
            return []
        result = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            piece = text[start:end]
            result.append(piece)
            start += chunk_size - overlap
        return result

    i = 0
    while i < len(lines):
        line = lines[i]

        # Track the most recent markdown header — used as context for tables
        if line.strip().startswith('#'):
            current_header = line.strip()
            buffer.append(line)
            i += 1
            continue

        # Detect start of a markdown table (line starts with '|')
        if line.strip().startswith('|'):
            # Flush whatever plain text was accumulated so far
            for piece in flush_text_buffer(buffer):
                chunks.append({"text": piece, "type": "text"})
            buffer = []

            # Collect the full table block (consecutive lines starting with '|')
            table_lines = []
            while i < len(lines) and lines[i].strip().startswith('|'):
                table_lines.append(lines[i])
                i += 1

            table_text = '\n'.join(table_lines)
            # Prepend the most recent header so the table is self-contained
            full_table_chunk = f"{current_header}\n\n{table_text}".strip()
            chunks.append({"text": full_table_chunk, "type": "table"})
            continue

        buffer.append(line)
        i += 1

    # Flush any remaining text
    for piece in flush_text_buffer(buffer):
        chunks.append({"text": piece, "type": "text"})

    return chunks

# Process all documents in the parsed_docs dictionary
all_chunks = {}

for filename, markdown_content in parsed_docs.items():
    chunks = smart_chunk_markdown(markdown_content, chunk_size=500, overlap=50)
    all_chunks[filename] = chunks

    # Print summary for each document
    table_count = sum(1 for c in chunks if c["type"] == "table")
    text_count = sum(1 for c in chunks if c["type"] == "text")
    print(f"✅ {filename}: {len(chunks)} total chunks ({table_count} tables, {text_count} text)")

print(f"\n Total chunks across all documents: {sum(len(c) for c in all_chunks.values())}")

# Preview one table chunk from Tesla to verify it kept its header + stayed whole
print("\n--- Example Tesla table chunk preview ---\n")
tesla_chunks = all_chunks["TSLA-Q4-2025-Update.pdf"]
tesla_tables = [c for c in tesla_chunks if c["type"] == "table"]
if tesla_tables:
    print(tesla_tables[0]["text"][:600])
else:
    print("No table chunks found in Tesla document.")

✅ apple_10k_2025.pdf: 615 total chunks (48 tables, 567 text)
✅ TSLA-Q4-2025-Update.pdf: 79 total chunks (16 tables, 63 text)

 Total chunks across all documents: 694

--- Example Tesla table chunk preview ---

| Highlights                |   03 |
|---------------------------|------|
| Financial Summary         |   04 |
| Operational Summary       |   06 |
| Manufacturing & Hardware  |   08 |
| Supporting Infrastructure |   09 |
| AI & Software             |   10 |
| Services                  |   11 |
| Other Updates             |   12 |
| Outlook                   |   13 |
| Photos & Charts           |   14 |
| Key Metrics               |   24 |
| Financial Statements      |   27 |
| Additional Information    |   34 |


In [10]:
# PHASE 1 - Cell 4: Generate embeddings for all documents
# We embed all chunks in one batch for efficiency, keeping track of which
# document each chunk belongs to (for proper user_id mapping later).

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Flatten all chunks into a single list, keeping track of metadata
all_texts = []
chunk_metadata = []  # (filename, chunk_index, chunk_type)

for filename, chunks in all_chunks.items():
    for idx, chunk in enumerate(chunks):
        all_texts.append(chunk["text"])
        chunk_metadata.append((filename, idx, chunk["type"]))

print(f"Total chunks to embed: {len(all_texts)}")

# Generate embeddings in one batch
print("Generating embeddings...")
embeddings = model.encode(all_texts, show_progress_bar=True, batch_size=32)

print(f"\n✅ Embeddings generated. Shape: {embeddings.shape}")
print(f"   (694 chunks × 384 dimensions)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Total chunks to embed: 694
Generating embeddings...


Batches:   0%|          | 0/22 [00:00<?, ?it/s]


✅ Embeddings generated. Shape: (694, 384)
   (694 chunks × 384 dimensions)


In [11]:
# PHASE 1 - Cell 5 (FIX): Create payload index on user_id for filtering
from qdrant_client.models import PayloadSchemaType

qdrant.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="user_id",
    field_schema=PayloadSchemaType.KEYWORD
)

print(f"✅ Index created on 'user_id' field for '{COLLECTION_NAME}'.")

✅ Index created on 'user_id' field for 'vaultify_documents'.


In [12]:
# PHASE 1 - Cell 6: Upload all chunks to Qdrant with deterministic IDs and proper user mapping
# We map each document to its corresponding user_id for multi-tenant isolation:
#   - apple_10k_2025.pdf → test_user_apple_demo (company_a)
#   - TSLA-Q4-2025-Update.pdf → test_user_tesla_demo (company_b)

from qdrant_client.models import VectorParams, Distance, PointStruct
import hashlib

COLLECTION_NAME = "vaultify_documents"
VECTOR_SIZE = 384  # matches all-MiniLM-L6-v2 output

# Map filenames to user_ids for multi-tenant isolation
USER_MAPPING = {
    "apple_10k_2025.pdf": "test_user_apple_demo",
    "TSLA-Q4-2025-Update.pdf": "test_user_tesla_demo"
}

def deterministic_id(filename, chunk_index):
    """Generate a stable ID from filename + chunk_index, so re-running this
    cell updates existing points instead of creating duplicates."""
    raw = f"{filename}_{chunk_index}"
    return hashlib.md5(raw.encode()).hexdigest()

# Create the collection (skip if it already exists)
existing_collections = [c.name for c in qdrant.get_collections().collections]

if COLLECTION_NAME not in existing_collections:
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
    )
    print(f"✅ Collection '{COLLECTION_NAME}' created.")
else:
    print(f"ℹ️ Collection '{COLLECTION_NAME}' already exists, skipping creation.")

# Build the points (vector + payload) for each chunk
points = []
for idx, (text, vector, (filename, chunk_idx, chunk_type)) in enumerate(
    zip(all_texts, embeddings, chunk_metadata)
):
    user_id = USER_MAPPING.get(filename)
    if not user_id:
        print(f"⚠️ Warning: No user mapping for {filename}, skipping chunk {chunk_idx}")
        continue

    point = PointStruct(
        id=deterministic_id(filename, chunk_idx),
        vector=vector.tolist(),
        payload={
            "user_id": user_id,
            "filename": filename,
            "chunk_type": chunk_type,
            "text": text,
            "chunk_index": chunk_idx
        }
    )
    points.append(point)

print(f"Prepared {len(points)} points for upload.")

# Upload in batches (Qdrant recommends batching for large uploads)
batch_size = 100
for i in range(0, len(points), batch_size):
    batch = points[i:i+batch_size]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=batch)
    print(f"  Uploaded batch {i // batch_size + 1} ({len(batch)} points)")

print(f"\n✅ All {len(points)} chunks uploaded to Qdrant.")

# Verify by counting points in the collection
collection_info = qdrant.get_collection(COLLECTION_NAME)
print(f"Total points in collection now: {collection_info.points_count}")

# Verify user distribution
apple_count = qdrant.count(
    collection_name=COLLECTION_NAME,
    count_filter={"must": [{"key": "user_id", "match": {"value": "test_user_apple_demo"}}]}
).count
tesla_count = qdrant.count(
    collection_name=COLLECTION_NAME,
    count_filter={"must": [{"key": "user_id", "match": {"value": "test_user_tesla_demo"}}]}
).count

print(f"\n📊 User distribution:")
print(f"   test_user_apple_demo: {apple_count} chunks")
print(f"   test_user_tesla_demo: {tesla_count} chunks")

ℹ️ Collection 'vaultify_documents' already exists, skipping creation.
Prepared 694 points for upload.
  Uploaded batch 1 (100 points)
  Uploaded batch 2 (100 points)
  Uploaded batch 3 (100 points)
  Uploaded batch 4 (100 points)
  Uploaded batch 5 (100 points)
  Uploaded batch 6 (100 points)
  Uploaded batch 7 (94 points)

✅ All 694 chunks uploaded to Qdrant.
Total points in collection now: 1309

📊 User distribution:
   test_user_apple_demo: 1230 chunks
   test_user_tesla_demo: 79 chunks


In [13]:
# PHASE 2 - Cell 1: Define the MCP server and search tool
# We explicitly define TEST_USER_ID here to ensure the MCP server
# has access to it regardless of notebook execution order.

from mcp.server.fastmcp import FastMCP
from mcp.server.transport_security import TransportSecuritySettings

# Explicitly define the test user ID for MVP stage
TEST_USER_ID = "test_user_apple_demo"

# Initialize FastMCP with DNS rebinding protection disabled for Ngrok
mcp = FastMCP(
    "Vaultify",
    host="0.0.0.0",
    port=8000,
    transport_security=TransportSecuritySettings(
        enable_dns_rebinding_protection=False,
    )
)

@mcp.tool()
def search_documents(query: str) -> str:
    """
    Search the company's uploaded documents and return relevant context
    to answer the user's question. Use this whenever the user asks about
    information that might be in their uploaded files (financial data,
    reports, contracts, etc.).

    Args:
        query: The user's question or search query

    Returns:
        Relevant document excerpts as context for answering the question
    """
    # Embed the query using the same model used for indexing
    query_vector = model.encode(query).tolist()

    # Search Qdrant with user_id filter (ensures multi-tenant isolation)
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter={
            "must": [{"key": "user_id", "match": {"value": TEST_USER_ID}}]
        },
        limit=8  # Wide-net strategy: pull more candidates, let LLM pick the right one
    ).points

    if not results:
        return "No relevant information found in the uploaded documents."

    # Format context for the LLM
    context_parts = [r.payload['text'] for r in results]
    return "\n\n---\n\n".join(context_parts)

print("✅ MCP server 'Vaultify' defined with tool: search_documents")

✅ MCP server 'Vaultify' defined with tool: search_documents


In [14]:
# PHASE 2 - Cell 2: Run MCP server and create Ngrok tunnel
import threading
import time
from pyngrok import ngrok

# Close any existing tunnels first
ngrok.kill()

# Run the MCP server in a background thread
def run_server():
    mcp.run(transport="sse")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

# Create Ngrok tunnel
ngrok.set_auth_token(NGROK_TOKEN)
public_tunnel = ngrok.connect(8000)

print(f"✅ Ngrok tunnel opened: {public_tunnel.public_url}")
print(f"👉 SSE endpoint: {public_tunnel.public_url}/sse")

INFO:     Started server process [17719]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Ngrok tunnel opened: https://rotten-tipping-laborer.ngrok-free.dev
👉 SSE endpoint: https://rotten-tipping-laborer.ngrok-free.dev/sse


In [15]:
# PHASE 3 - Cell 1: Create folder structure for templates
import os

os.makedirs('/content/templates', exist_ok=True)
os.makedirs('/content/uploads', exist_ok=True)

print("✅ Folder structure created.")

✅ Folder structure created.


In [16]:
# PHASE 3 - Cell 2: templates/login.html
login_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Vaultify - Login</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
    <style>
        body { font-family: 'Inter', sans-serif; }
    </style>
</head>
<body class="bg-zinc-950 h-screen flex items-center justify-center">

    <div class="w-full max-w-sm px-6">
        <!-- Logo / brand -->
        <div class="flex items-center gap-2 mb-8 justify-center">
            <div class="w-8 h-8 rounded-lg bg-indigo-500 flex items-center justify-center">
                <svg class="w-5 h-5 text-white" fill="none" stroke="currentColor" viewBox="0 0 24 24">
                    <path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M12 15v2m-6 4h12a2 2 0 002-2v-6a2 2 0 00-2-2H6a2 2 0 00-2 2v6a2 2 0 002 2zm10-10V7a4 4 0 00-8 0v4h8z"></path>
                </svg>
            </div>
            <span class="text-zinc-100 font-semibold text-xl tracking-tight">Vaultify</span>
        </div>

        <div class="bg-zinc-900 border border-zinc-800 rounded-xl p-8">
            <h1 class="text-zinc-100 text-lg font-semibold mb-1">Sign in</h1>
            <p class="text-zinc-500 text-sm mb-6">Document intelligence for your business</p>

            {% if error %}
            <div class="bg-red-950 border border-red-900 text-red-400 px-3 py-2 rounded-md mb-4 text-sm">
                {{ error }}
            </div>
            {% endif %}

            <form method="POST" action="/login">
                <div class="mb-4">
                    <label class="block text-zinc-400 text-xs font-medium mb-2">Username</label>
                    <input type="text" name="username" required
                           class="w-full bg-zinc-950 border border-zinc-800 rounded-md px-3 py-2 text-sm text-zinc-100
                                  placeholder-zinc-600 focus:outline-none focus:ring-1 focus:ring-indigo-500 focus:border-indigo-500
                                  transition-colors">
                </div>
                <div class="mb-6">
                    <label class="block text-zinc-400 text-xs font-medium mb-2">Password</label>
                    <input type="password" name="password" required
                           class="w-full bg-zinc-950 border border-zinc-800 rounded-md px-3 py-2 text-sm text-zinc-100
                                  placeholder-zinc-600 focus:outline-none focus:ring-1 focus:ring-indigo-500 focus:border-indigo-500
                                  transition-colors">
                </div>
                <button type="submit"
                        class="w-full bg-indigo-500 hover:bg-indigo-400 text-white text-sm font-medium py-2 rounded-md
                               transition-colors">
                    Sign In
                </button>
            </form>
        </div>

        <p class="text-zinc-600 text-xs text-center mt-6">
            Vaultify &middot; Multi-tenant document RAG
        </p>
    </div>

</body>
</html>
"""

with open('/content/templates/login.html', 'w') as f:
    f.write(login_html)

print("✅ login.html created.")

✅ login.html created.


In [17]:
# PHASE 3 - Cell 3: templates/dashboard.html
dashboard_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Vaultify - Dashboard</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
    <style>
        body { font-family: 'Inter', sans-serif; }
    </style>
</head>
<body class="bg-zinc-950 min-h-screen">

    <!-- Top nav -->
    <nav class="border-b border-zinc-800 px-6 py-4 flex justify-between items-center">
        <div class="flex items-center gap-2">
            <div class="w-7 h-7 rounded-lg bg-indigo-500 flex items-center justify-center">
                <svg class="w-4 h-4 text-white" fill="none" stroke="currentColor" viewBox="0 0 24 24">
                    <path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M12 15v2m-6 4h12a2 2 0 002-2v-6a2 2 0 00-2-2H6a2 2 0 00-2 2v6a2 2 0 002 2zm10-10V7a4 4 0 00-8 0v4h8z"></path>
                </svg>
            </div>
            <span class="text-zinc-100 font-semibold text-sm tracking-tight">Vaultify</span>
        </div>
        <div class="flex items-center gap-4">
            <span class="text-sm text-zinc-500">{{ username }}</span>
            <a href="/logout" class="text-sm text-zinc-500 hover:text-zinc-300 transition-colors">Logout</a>
        </div>
    </nav>

    <div class="max-w-3xl mx-auto mt-10 px-6 pb-12">

        <!-- Upload section -->
        <div class="bg-zinc-900 border border-zinc-800 rounded-xl p-6 mb-5">
            <h2 class="text-zinc-100 text-sm font-semibold mb-1">Upload a document</h2>
            <p class="text-zinc-500 text-xs mb-4">PDF files are parsed, chunked, and indexed automatically.</p>

            <form method="POST" action="/upload" enctype="multipart/form-data" class="flex gap-3">
                <input type="file" name="file" accept=".pdf" required
                       class="flex-1 bg-zinc-950 border border-zinc-800 rounded-md px-3 py-2 text-sm text-zinc-300
                              file:mr-3 file:py-1 file:px-3 file:rounded file:border-0 file:bg-zinc-800
                              file:text-zinc-300 file:text-xs file:font-medium
                              focus:outline-none focus:ring-1 focus:ring-indigo-500">
                <button type="submit"
                        class="bg-indigo-500 hover:bg-indigo-400 text-white text-sm font-medium px-5 py-2 rounded-md
                               transition-colors whitespace-nowrap">
                    Upload
                </button>
            </form>

            {% if upload_status %}
            <div class="mt-3 text-sm {{ 'text-emerald-400' if upload_success else 'text-red-400' }}">
                {{ upload_status }}
            </div>
            {% endif %}
        </div>

        <!-- Q&A section -->
        <div class="bg-zinc-900 border border-zinc-800 rounded-xl p-6">
            <h2 class="text-zinc-100 text-sm font-semibold mb-1">Ask a question</h2>
            <p class="text-zinc-500 text-xs mb-4">Ask anything about your uploaded documents.</p>

            <form method="POST" action="/ask" class="flex gap-3 mb-2">
                <input type="text" name="question" placeholder="e.g. What were the total net sales in 2025?"
                       class="flex-1 bg-zinc-950 border border-zinc-800 rounded-md px-3 py-2 text-sm text-zinc-100
                              placeholder-zinc-600 focus:outline-none focus:ring-1 focus:ring-indigo-500 focus:border-indigo-500"
                       required>
                <button type="submit"
                        class="bg-zinc-800 hover:bg-zinc-700 text-zinc-100 text-sm font-medium px-5 py-2 rounded-md
                               transition-colors whitespace-nowrap">
                    Ask
                </button>
            </form>

            {% if answer %}
            <div class="bg-zinc-950 border border-zinc-800 rounded-md p-4 mt-4">
                <p class="text-xs text-zinc-500 mb-1">Question</p>
                <p class="text-zinc-300 text-sm mb-3">{{ question }}</p>
                <p class="text-xs text-zinc-500 mb-1">Answer</p>
                <p class="text-zinc-100 text-sm leading-relaxed">{{ answer }}</p>
            </div>
            {% endif %}
        </div>

    </div>

</body>
</html>
"""

with open('/content/templates/dashboard.html', 'w') as f:
    f.write(dashboard_html)

print("✅ dashboard.html created.")

✅ dashboard.html created.


In [18]:
# PHASE 3 - Cell 4 (COMPLETE): Flask app + routes + document processing + Q&A

from flask import Flask, request, render_template, redirect, session, url_for
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, PayloadSchemaType
import hashlib

# ===== Load dependencies ONCE at module level =====
print("🔄 Loading dependencies...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")

QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("✅ Qdrant connected")

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client ready")

COLLECTION_NAME = "vaultify_documents"
print("✅ All dependencies initialized\n")

# ===== Flask app setup =====
app = Flask(__name__, template_folder='/content/templates')
app.secret_key = "dev_secret_key_change_in_production"

# Test users for multi-tenant testing
USERS = {
    "company_a": {"password": "demo123", "user_id": "test_user_apple_demo"},
    "company_b": {"password": "demo456", "user_id": "test_user_tesla_demo"},
}

# ===== Routes =====
@app.route('/')
def index():
    if 'username' not in session:
        return redirect(url_for('login_page'))
    return redirect(url_for('dashboard'))

@app.route('/login', methods=['GET'])
def login_page():
    return render_template('login.html')

@app.route('/login', methods=['POST'])
def login():
    username = request.form.get('username')
    password = request.form.get('password')

    user = USERS.get(username)
    if user and user['password'] == password:
        session['username'] = username
        session['user_id'] = user['user_id']
        return redirect(url_for('dashboard'))
    else:
        return render_template('login.html', error="Invalid username or password")

@app.route('/logout')
def logout():
    session.clear()
    return redirect(url_for('login_page'))

@app.route('/dashboard')
def dashboard():
    if 'username' not in session:
        return redirect(url_for('login_page'))
    return render_template('dashboard.html', username=session['username'])

@app.route('/upload', methods=['POST'])
def upload():
    if 'username' not in session:
        return redirect(url_for('login_page'))

    file = request.files.get('file')
    if not file or file.filename == '':
        return render_template('dashboard.html', username=session['username'],
                                upload_status="No file selected.", upload_success=False)

    save_path = f"/content/uploads/{file.filename}"
    file.save(save_path)

    try:
        n_chunks = process_and_index_document(save_path, file.filename, session['user_id'])
        return render_template('dashboard.html', username=session['username'],
                                upload_status=f"✅ '{file.filename}' uploaded ({n_chunks} chunks).",
                                upload_success=True)
    except Exception as e:
        return render_template('dashboard.html', username=session['username'],
                                upload_status=f"❌ Error: {str(e)[:100]}", upload_success=False)

@app.route('/ask', methods=['POST'])
def ask():
    if 'username' not in session:
        return redirect(url_for('login_page'))

    question = request.form.get('question')

    # DEBUG: Check session contents
    print(f"\n[DEBUG /ask] Session contents: {dict(session)}")
    print(f"[DEBUG /ask] user_id from session: {session.get('user_id')}")

    user_id = session.get('user_id')
    if not user_id:
        return render_template('dashboard.html', username=session['username'],
                                question=question,
                                answer="ERROR: user_id not found in session. Please logout and login again.")

    answer = answer_question(question, user_id)

    return render_template('dashboard.html', username=session['username'],
                            question=question, answer=answer)

# ===== Document processing functions =====
def process_and_index_document(pdf_path, filename, user_id):
    """Process a PDF: parse, chunk, embed, upload to Qdrant."""
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
    from docling.datamodel.base_models import InputFormat
    import torch

    # Parse with Docling
    pipeline_options = PdfPipelineOptions()
    if torch.cuda.is_available():
        pipeline_options.accelerator_options = AcceleratorOptions(num_threads=4, device=AcceleratorDevice.CUDA)

    converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)})
    result = converter.convert(pdf_path)
    markdown_content = result.document.export_to_markdown()

    # Smart chunking
    def smart_chunk_markdown(markdown_text, chunk_size=500, overlap=50):
        chunks = []
        lines = markdown_text.split('\n')
        current_header = ""
        buffer = []

        def flush_text_buffer(buffer_lines):
            text = '\n'.join(buffer_lines).strip()
            if not text:
                return []
            result = []
            start = 0
            while start < len(text):
                result.append(text[start:start+chunk_size])
                start += chunk_size - overlap
            return result

        i = 0
        while i < len(lines):
            line = lines[i]
            if line.strip().startswith('#'):
                current_header = line.strip()
                buffer.append(line)
                i += 1
                continue
            if line.strip().startswith('|'):
                for piece in flush_text_buffer(buffer):
                    chunks.append({"text": piece, "type": "text"})
                buffer = []
                table_lines = []
                while i < len(lines) and lines[i].strip().startswith('|'):
                    table_lines.append(lines[i])
                    i += 1
                table_text = '\n'.join(table_lines)
                chunks.append({"text": f"{current_header}\n\n{table_text}".strip(), "type": "table"})
                continue
            buffer.append(line)
            i += 1

        for piece in flush_text_buffer(buffer):
            chunks.append({"text": piece, "type": "text"})

        return chunks

    chunks = smart_chunk_markdown(markdown_content, chunk_size=500, overlap=50)

    # Embedding
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts, show_progress_bar=False, batch_size=32)

    # Upload to Qdrant
    def deterministic_id(filename, chunk_index):
        return hashlib.md5(f"{filename}_{chunk_index}".encode()).hexdigest()

    points = []
    for idx, (chunk, vector) in enumerate(zip(chunks, embeddings)):
        point = PointStruct(
            id=deterministic_id(filename, idx),
            vector=vector.tolist(),
            payload={
                "user_id": user_id,
                "filename": filename,
                "chunk_type": chunk["type"],
                "text": chunk["text"],
                "chunk_index": idx
            }
        )
        points.append(point)

    batch_size = 100
    for i in range(0, len(points), batch_size):
        qdrant.upsert(collection_name=COLLECTION_NAME, points=points[i:i+batch_size])

    try:
        qdrant.create_payload_index(
            collection_name=COLLECTION_NAME,
            field_name="user_id",
            field_schema=PayloadSchemaType.KEYWORD
        )
    except:
        pass

    return len(chunks)


def answer_question(question, user_id):
    """Answer a question using global dependencies."""
    try:
        query_vector = model.encode(question).tolist()
        results = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            query_filter={
                "must": [{"key": "user_id", "match": {"value": user_id}}]
            },
            limit=8
        ).points

        if not results:
            return "No relevant information found in the uploaded documents."

        context = "\n\n---\n\n".join([r.payload['text'] for r in results])

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "Answer the user's question using ONLY the provided context. If the answer isn't in the context, say so clearly."},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
            ],
            max_tokens=300,
        )

        return response.choices[0].message.content

    except Exception as e:
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}"

print("✅ Flask app COMPLETE with routes, document processing, and Q&A.")

🔄 Loading dependencies...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded
✅ Qdrant connected
✅ Groq client ready
✅ All dependencies initialized

✅ Flask app COMPLETE with routes, document processing, and Q&A.


In [19]:
# PHASE 3 - Cell 5: Run Flask app and create Ngrok tunnel
import threading
import time
from pyngrok import ngrok

FLASK_PORT = 5001  # Use different port to avoid conflicts

def run_flask():
    app.run(host="0.0.0.0", port=FLASK_PORT, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

time.sleep(3)

# Close existing tunnels and create new one
ngrok.kill()
flask_tunnel = ngrok.connect(FLASK_PORT)

print(f"✅ Flask app running on port {FLASK_PORT}")
print(f"✅ Public URL: {flask_tunnel.public_url}")
print(f"\n👉 Open this link in your browser: {flask_tunnel.public_url}")
print(f"   Login with:")
print(f"   - company_a / demo123 (Apple documents)")
print(f"   - company_b / demo456 (Tesla documents)")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit


✅ Flask app running on port 5001
✅ Public URL: https://rotten-tipping-laborer.ngrok-free.dev

👉 Open this link in your browser: https://rotten-tipping-laborer.ngrok-free.dev
   Login with:
   - company_a / demo123 (Apple documents)
   - company_b / demo456 (Tesla documents)
